In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM-135M"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

text = "What is the capital of France?"
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=20, use_cache=False)
print(tokenizer.decode(outputs, skip_special_tokens=True)[0])

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

What is the capital of France?
Paris, the capital of France, is the largest city in the country and the second-largest


In [ ]:
import torch

from transformers.models.llama.modeling_llama import LlamaAttention
import torch
import torch.nn.functional as F
from transformers import DynamicCache
import math

def forward(
    self,
    hidden_states: torch.Tensor,
    position_embeddings: tuple[torch.Tensor, torch.Tensor] = None,
    attention_mask: torch.Tensor = None,
    past_key_values: DynamicCache = None,
    **kwargs,
) -> tuple[torch.Tensor, torch.Tensor | None]:
    B, S, _ = hidden_states.shape
    D = self.head_dim

    query = self.q_proj(hidden_states).reshape(B, S, -1, D).transpose(1, 2)
    key = self.k_proj(hidden_states).reshape(B, S, -1, D).transpose(1, 2)
    value = self.v_proj(hidden_states).reshape(B, S, -1, D).transpose(1, 2)

    cos, sin = position_embeddings

    def rope(x):
        return x * cos + torch.cat([-x[..., D // 2:], x[..., :D // 2]], dim=-1) * sin

    query = rope(query)
    key = rope(key)

    if past_key_values is not None:
        # appends key value chunk and gets full key value
        key, value = past_key_values.update(key, value, self.layer_idx)

        if attention_mask is None:
            S_q = query.shape[2]
            S_kv = key.shape[2]
            attention_mask = torch.triu(
                torch.full((B, 1, S_q, S_kv), -math.inf, device=query.device, dtype=query.dtype),
                diagonal=1 + S_kv - S_q,
            )

    output = F.scaled_dot_product_attention(
        query, key, value,
        attn_mask=attention_mask,
        is_causal=attention_mask is None,
        scale=self.scaling,
        enable_gqa=True,
    )
    return self.o_proj(output.transpose(1, 2).reshape(B, S, -1)), None

LlamaAttention.forward = forward

outputs = model.generate(**inputs, max_new_tokens=20, use_cache=True)
print(tokenizer.decode(outputs, skip_special_tokens=True)[0])

What is the capital of France?
Paris, the capital of France, is the largest city in the country and the second-largest
